# VDC Surrogate Model\n\nLoad the trained neural network surrogate and use it to predict VDC performance.\nEnables brute-force evaluation of millions of calibrations in seconds.

In [ ]:
import torch\nimport numpy as np\n\ndata = torch.load('models/vdc_surrogate_v2.pt', map_location='cpu')\nmodel = torch.nn.Sequential(\n    torch.nn.Linear(6, 128), torch.nn.ReLU(),\n    torch.nn.Linear(128, 128), torch.nn.ReLU(),\n    torch.nn.Linear(128, 128), torch.nn.ReLU(),\n    torch.nn.Linear(128, 3)\n)\nmodel.load_state_dict(data['model'])\nmodel.eval()\n\nx = torch.tensor([[1500.0, 60.0, 0.08, 0.17, 0.97, 0.58]])\nx_n = (x - data['X_mean']) / data['X_std']\nwith torch.no_grad():\n    pred = model(x_n) * data['y_std'] + data['y_mean']\nprint(f'Predicted: lap={pred[0,0]:.2f}s  yaw_err={pred[0,1]:.4f}  energy={pred[0,2]:.0f}J')\nprint(f'Inference time: microseconds')

In [ ]:
# Brute-force 1M calibrations in under 1 second\nn = 1_000_000\nX = torch.rand(n, 6)\nbounds = torch.tensor([[500,2000],[0,60],[0.05,0.20],[0.1,1.0],[0,1],[0.55,0.70]])\nX_s = X * (bounds[:,1] - bounds[:,0]) + bounds[:,0]\nX_n = (X_s - data['X_mean']) / data['X_std']\n\nwith torch.no_grad():\n    preds = model(X_n) * data['y_std'] + data['y_mean']\n\nbest = preds[:,0].argmin()\nprint(f'Best of {n:,}: lap={preds[best,0]:.2f}s')\nprint(f'yaw_g={X_s[best,0]:.0f} yaw_d={X_s[best,1]:.0f}')\nprint(f'Evaluated {n:,} surrogate predictions in under 1 second')